In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import itertools


# independent crash prob

In [ ]:
def simulate_crashes(num_cars, crash_prob_per_sec, max_seconds=7000, seed=None):
    """
    Simulate car crashes over time.
    Each second, each car has a crash_prob_per_sec chance to crash.
    Returns a list of (car_id, crash_time) for each crash.
    """
    rng = np.random.default_rng(seed)
    crash_events = []
    # Track which cars have crashed
    crashed = set()
    for t in range(max_seconds):
        for car_id in range(num_cars):
            if car_id in crashed:
                continue
            if rng.random() < crash_prob_per_sec:
                crash_events.append((car_id, t))
                crashed.add(car_id)
    return crash_events

# Example usage:
num_cars = 1000
crash_prob_per_sec = 0.00001  # Adjust this value to test
crashes = simulate_crashes(num_cars, crash_prob_per_sec)

print(f"Total crashes: {len(crashes)}")


In [ ]:
crash_prob = [1,2,3,4,5] 
n_cars = [100, 200, 500]
avg_time_till_top = [20, 30, 45, 60]

results = []

for p in crash_prob:
    for n in n_cars:
        for t in avg_time_till_top:
            crashes = (t * 60) * n * ((p/15)**.5)/800000
            results.append({'crash_prob': p, 'n_cars': n, 'avg_time_till_top': t,'vmt':(11.6*n), 'expected_crashes': crashes})

df = pd.DataFrame(results)
df['crashes_per_100k_vmt'] = df['expected_crashes'] / (df['vmt'] / 100000)
df.head()





In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # 1 row, 3 columns

for ax, col in zip(axes, ['crash_prob',  'avg_time_till_top']):
    sns.scatterplot(x=df[col], y=df['crashes_per_100k_vmt'], ax=ax)
    ax.set_title(f"{col} vs. crashes_per_100k_vmt")
    ax.set_xlabel(col)
    ax.set_ylabel("crashes_per_100k_vmt")

plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(df['crashes_per_100k_vmt'], kde=True)

In [ ]:

def should_crash_happen(crashes_per_100k_vmt, num_cars, avg_speed_mps):
    """
    Returns True if a crash should occur this step, based on the desired crash rate
    and instantaneous traffic conditions.
    """
    # Convert crashes per 100k miles to crashes per meter
    METERS_PER_100K_MILES = 160934400  # 100,000 miles in meters
    crashes_per_meter = crashes_per_100k_vmt / METERS_PER_100K_MILES

    # Estimate how many meters were driven this second
    meters_driven = num_cars * avg_speed_mps

    # Expected number of crashes this step
    expected_crashes = crashes_per_meter * meters_driven

    # Use Poisson process to determine if a crash happens
    return np.random.poisson(expected_crashes)
    
def test_crash_matrix(crashes_per_100k_vmt_list, num_cars_list, avg_mps_list, run_steps_list):
    results = []
    METERS_PER_100K_MILES = 160934400  # 100,000 miles in meters

    for crashes_per_100k_vmt in crashes_per_100k_vmt_list:
        for num_cars in num_cars_list:
            for avg_mps in avg_mps_list:
                for run_steps in run_steps_list:
                    total_crashes = 0
                    for _ in range(run_steps):
                        # Call your crash function for each step
                        crashes_this_step = should_crash_happen(crashes_per_100k_vmt, num_cars, avg_mps)
                        total_crashes += crashes_this_step

                    # Total VMT in meters
                    total_vmt_m = num_cars * avg_mps * run_steps
                    # Actual crashes per 100k VMT
                    actual_crashes_per_100k_vmt = total_crashes / (total_vmt_m / METERS_PER_100K_MILES)

                    results.append({
                        'crashes_per_100k_vmt_input': crashes_per_100k_vmt,
                        'num_cars': num_cars,
                        'avg_mps': avg_mps,
                        'run_steps': run_steps,
                        'total_crashes': total_crashes,
                        'total_vmt_m': total_vmt_m,
                        'actual_crashes_per_100k_vmt': actual_crashes_per_100k_vmt
                    })

    return pd.DataFrame(results)



In [ ]:
# Example usage of the test function
crashes_per_100k_vmt = 91000 # tzhis will be the input
num_cars = 300 # this will be a model output based on number of cars on the road
average_mps = 30 # this will be a model output based on the speed of not stopped cars


should_crash_happen(crashes_per_100k_vmt, num_cars, average_mps)

# Cumulative crash function

this is the superior approach because it results in much less error in small samples

## this is the main function that will be used in the model


In [ ]:
import numpy as np

import numpy as np

def should_crash_happen_cumulative(
    crashes_per_100k_vmt: float,
    num_cars: int,
    avg_speed_mps: float,
    *,
    method: str = "randomized_rounding",      # "randomized_rounding" | "poisson" | "batched_poisson"
    batch_seconds: int | None = None,         # window size in steps (since 1 step = 1 s)
    state: dict | None = None,
    rng: np.random.Generator | None = None
):
    """
    Returns:
        crashes: int (>=0)
        state: dict (persist between calls)

    State fields:
        - 'tracker': float
        - 'batch_elapsed_steps': int
        - 'batch_lam': float
    """
    METERS_PER_100K_MILES = 160_934_400.0
    if rng is None:
        rng = np.random.default_rng()
    if state is None:
        state = {}
    state.setdefault("tracker", 0.0)
    state.setdefault("batch_elapsed_steps", 0)
    state.setdefault("batch_lam", 0.0)

    # Exposure for this step (meters), since 1 step = 1 second
    meters_this_step = float(num_cars) * float(avg_speed_mps)
    lam_step = (crashes_per_100k_vmt / METERS_PER_100K_MILES) * meters_this_step

    if method == "poisson_clock":
        # state: 'r' = remaining intensity until next event, Exp(1) distributed
        if "r" not in state or state["r"] <= 0:
            state["r"] = rng.exponential(1.0)
        r = state["r"] - lam_step
        crashes = 0
        while r <= 0.0:
            crashes += 1
            r += rng.exponential(1.0)  # schedule next event in intensity units
        state["r"] = r

    elif method == "randomized_rounding":
        state["tracker"] += lam_step
        base = int(state["tracker"])
        frac = state["tracker"] - base
        crashes = max(base, (1 if rng.random() < frac else 0))
        state["tracker"] -= crashes

    elif method == "poisson":
        crashes = int(rng.poisson(lam_step))

    elif method == "batched_poisson":
        if batch_seconds is None or batch_seconds <= 0:
            raise ValueError("For method='batched_poisson', provide a positive batch_seconds.")
        state["batch_elapsed_steps"] += 1
        state["batch_lam"] += lam_step
        if state["batch_elapsed_steps"] >= int(batch_seconds):
            crashes = int(rng.poisson(state["batch_lam"]))
            state["batch_elapsed_steps"] = 0
            state["batch_lam"] = 0.0
        else:
            crashes = 0
    else:
        raise ValueError("method must be one of {'randomized_rounding','poisson','batched_poisson'}")

    return crashes, state


##### Example usage of the test function

# randomized_rounding 
# state = {"tracker": 0.0}
# crashes, state = should_crash_happen_cumulative(rate, n_cars, avg_mps, method='randomized_rounding', state=state)

# poisson
# crashes, state = should_crash_happen_cumulative(rate, n_cars, avg_mps, method="poisson", state=state)

# batched_poisson
# state = {"batch_elapsed_s": 0.0, "batch_lam": 0.0}
# crashes, state = should_crash_happen_cumulative(rate, n_cars, avg_mps,
#                                                 method="batched_poisson",
#                                                 batch_seconds=30, state=state)


state = {"tracker": 0.0}
crashes, state = should_crash_happen_cumulative(22, 400, 20, method='randomized_rounding')

crashes

# Random rounding final function

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

def error_plot(df, x_col, y_col, lower_quant=0.25, upper_quant=0.75,
               split_methods=False, smooth_frac=0.2, iqr_color='blue'):
    df = df.copy()


    def _add_shading(ax, data):
        d = data.sort_values(x_col)
        # Guard: need at least a few points to compute quantiles & smooth
        if d.shape[0] < 5:
            return
        lower = d.groupby(x_col, as_index=False)[y_col].quantile(lower_quant)
        upper = d.groupby(x_col, as_index=False)[y_col].quantile(upper_quant)

        # Need matching x for fill_between; inner join on x
        q = lower.merge(upper, on=x_col, suffixes=('_lo', '_hi')).dropna()
        if q.shape[0] < 5:
            return

        sm_lo = lowess(q[f"{y_col}_lo"], q[x_col], frac=smooth_frac, return_sorted=False)
        sm_hi = lowess(q[f"{y_col}_hi"], q[x_col], frac=smooth_frac, return_sorted=False)
        ax.fill_between(q[x_col], sm_lo, sm_hi, color=iqr_color, alpha=0.2,
                        label=f'IQR ({int(100*lower_quant)}–{int(100*upper_quant)}%)')

    def _add_sse_box(ax, data):
        sse = float(np.square(data[y_col]).sum())
        txt = f"SSE = {sse:,.0f}"
        ax.text(0.02, 0.95, txt, transform=ax.transAxes, va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="gray"),
                fontsize=9)

    if not split_methods:
        plt.figure(figsize=(8, 4))
        ax = sns.scatterplot(
            data=df, x=x_col, y=y_col,
            alpha=0.7, s=30, edgecolor=None
        )
        ax.axhline(0, color='black', linestyle='-', linewidth=1)
        _add_shading(ax, df)
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title('Absolute Error vs Total VMT')
        plt.tight_layout()
        plt.show()
    else:
        g = sns.FacetGrid(df, col="method_label", sharey=True, sharex=False,
                          height=4, aspect=1.2, despine=True)

        def _facet(data, color, **kws):
            ax = plt.gca()
            sns.scatterplot(data=data, x=x_col, y=y_col, color=color,
                            alpha=0.7, s=30, edgecolor=None, ax=ax)
            ax.axhline(0, color='black', linestyle='-', linewidth=1)
            _add_shading(ax, data)
            _add_sse_box(ax, data)  # <- only called in split_methods=True path

        g.map_dataframe(_facet)
        g.set_axis_labels(x_col, y_col)
        g.set_titles(col_template="{col_name}")
        for ax in g.axes.flat:
            leg = ax.get_legend()
            if leg: leg.remove()
        plt.tight_layout()
        plt.show()








In [ ]:
def vmt_adjust(cum_vmt: int, modifier: float, crashes_per_100k_vmt: float, vmt_cutoff: int):
    '''
    Adjusts crashes_per_100k_vmt based on total_vmt and a modifier.
    If total_vmt is above a cutoff, returns crashes_per_100k_vmt unchanged.
    If below the cutoff, reduces crashes_per_100k_vmt proportionally to how far
    total_vmt is below the cutoff, scaled by the modifier.
    '''


    if cum_vmt > vmt_cutoff:
        return crashes_per_100k_vmt
    # else:
    #     pct_modifier_effect = 1-(cum_vmt/vmt_cutoff)
    #     reduction_multiplier =(1 - (pct_modifier_effect*modifier))
    #     adjusted_vmt = crashes_per_100k_vmt*reduction_multiplier
    #     #print(f'multiplier: {reduction_multiplier}, adjusted_vmt: {adjusted_vmt}')
    
    else:
        adjusted_vmt = modifier*crashes_per_100k_vmt
        return adjusted_vmt

vmt_adjust(400, .9, 22, 40000)


In [ ]:
def should_crash_capped_backlog_rr(
    cum_vmt :float,
    crashes_per_100k_vmt: float,
    num_cars: int,
    avg_speed_mps: float,
    remainder: float = 0.0,
    backlog: int = 0,
    rng: np.random.Generator | None = None
):
    """At most one per step; keeps totals right; adds Bernoulli(frac) when backlog==0."""
    if rng is None:
        rng = np.random.default_rng()
    crashes_per_100k_vmt_adjusted = vmt_adjust(cum_vmt=cum_vmt , modifier=.7, crashes_per_100k_vmt=crashes_per_100k_vmt, vmt_cutoff=30000) # slight downward bias to help keep totals right
    
    METERS_PER_100K_MILES = 160_934_400.0
    meters_this_step = float(num_cars) * float(avg_speed_mps)
    lam_step = (crashes_per_100k_vmt_adjusted / METERS_PER_100K_MILES) * meters_this_step

    inc = remainder + lam_step
    k = int(inc)
    frac = inc - k
    backlog += k

    if backlog > 0:
        crashes = 1
        backlog -= 1
        remainder = frac              # keep the fractional carry
    else:
        # No queued whole crash: do stochastic rounding on the fractional part
        u = rng.random()
        crashes = 1 if u < frac else 0
        remainder = inc - crashes     # (== frac - crashes when backlog==0)

    return crashes, remainder, backlog


In [ ]:
# this function is also represented in the more general function above. It is the randomized rounding method 1 for 1.

def should_crash_randomized_rounding(
    cum_vmt :float,
    crashes_per_100k_vmt: float,
    num_cars: int,
    avg_speed_mps: float,
    remainder: float = 0.0,
    rng: np.random.Generator | None = None
    ):
    """
    Randomized-rounding crash generator with no external state container.

    Args:
        crashes_per_100k_vmt: target crashes per 100,000 vehicle-miles.
        num_cars: cars on the road this step.
        avg_speed_mps: average speed (m/s) this step.
        remainder: fractional remainder carried from the previous step.
        rng: numpy random Generator (optional).

    Returns:
        crashes: int
        new_remainder: float (carry this to the next call)
    """
    METERS_PER_100K_MILES = 160_934_400.0
    if rng is None:
        rng = np.random.default_rng()

    meters_this_step = float(num_cars) * float(avg_speed_mps)
    lambda_step = (crashes_per_100k_vmt / METERS_PER_100K_MILES) * meters_this_step

    total = remainder + lambda_step # this continuiously increases until a crash happens then resets 
    base = int(total)
    frac = total - base
    crashes = base + (1 if rng.random() < frac else 0)
    new_remainder = total - crashes  # keep only the fractional part

    return crashes, new_remainder


## testing the crash function

### matrix of perameters test

In [ ]:
#Making the full test function that runs through a matrix of parameters and tests the cumulative crash function

def test_crash_matrix(crash_matrix_dict):
    # Create a DataFrame of all parameter combinations
    def dict_combinations_to_df(param_dict):
        # Remove 'samples' from the dict and get its value (default to 1 if not present)
        samples = param_dict.pop('samples', 1)
        keys = list(param_dict.keys())
        combos = list(itertools.product(*param_dict.values()))
        # Repeat each combo 'samples' times
        combos = combos * samples
        crash_param_df = pd.DataFrame(combos, columns=keys)
        print(f'number of samples tested: {len(crash_param_df)}')
        print(f'unique samples : {len(crash_param_df.drop_duplicates())}')
        return crash_param_df

    # create the actual dataframe of combinations
    crash_param_df = dict_combinations_to_df(crash_matrix_dict)


    # ------ run the sim here ------
    results = []
    METERS_PER_100K_MILES = 160_934_400.0
    M_PER_MILE = 1609.344

    for _, row in crash_param_df.iterrows():
        crash_counter = 0
        # ------- put the function perams here ------- 
        remainder = 0.0
        backlog = 0
        for i in range(int(row['run_steps'])):
            # # ----------------vvvvvvvvvvvvvvvvvvvvvvvvvvvvv define function here
            # crashes, remainder, backlog = should_crash_capped_backlog_rr(
            #     # these are the matrix stuff
            #     crashes_per_100k_vmt=float(row['crashes_per_100k_vmt']),
            #     num_cars=int(row['num_cars_per_step']),
            #     avg_speed_mps=float(row['avg_mps_of_step']),
            #     #for the spicific function 
            #     cum_vmt = row['avg_mps_of_step']*row['num_cars_per_step'] * i, 
            #     remainder = remainder,
            #     backlog = backlog

            # )

            crashes, remainder = should_crash_randomized_rounding(
                # these are the matrix stuff
                crashes_per_100k_vmt=float(row['crashes_per_100k_vmt']),
                num_cars=int(row['num_cars_per_step']),
                avg_speed_mps=float(row['avg_mps_of_step']),
                #for the spicific function 
                cum_vmt = row['avg_mps_of_step']*row['num_cars_per_step'] * i, 
                remainder = remainder
            )
            crash_counter += int(crashes)

        # Total VMT over the run (meters) = cars * m/s * seconds; seconds = run_steps
        total_vmt_m = float(row['num_cars_per_step']) * float(row['avg_mps_of_step']) * float(row['run_steps'])
        actual_crashes_per_100k_vmt = crash_counter / (total_vmt_m / METERS_PER_100K_MILES)
        expected_crashes = (float(row['crashes_per_100k_vmt']) / METERS_PER_100K_MILES) * total_vmt_m

        results.append({
            'actual_crashes_per_100k_vmt': actual_crashes_per_100k_vmt,
            'total_crashes': crash_counter,
            'total_vmt': total_vmt_m / M_PER_MILE,
            'expected_crashes': expected_crashes,
            # inputs
            'crashes_per_100k_vmt_input': float(row['crashes_per_100k_vmt']),
            'num_cars_input': int(row['num_cars_per_step']),
            'avg_mps_input': float(row['avg_mps_of_step']),
            'run_steps_input': int(row['run_steps']),
        })



    return pd.DataFrame(results)

In [ ]:


crash_matrix_dict = {
    "crashes_per_100k_vmt": [5, 10, 15, 20, 25, 50],
    "num_cars_per_step":    [100, 300],
    "avg_mps_of_step":      [10.0, 20.0, 30.0],
    "run_steps":            [ 3600, 7200, 14400],  # 1h, 2h, 4h at 1s steps
    "samples": 1
}

# Run the crash matrix test function
matrix_test_results = test_crash_matrix(crash_matrix_dict)

# Calculate errors
matrix_test_results['tot_crashes_error'] = matrix_test_results['total_crashes'] - matrix_test_results['expected_crashes']
matrix_test_results['tot_crashes_abs_error'] = matrix_test_results['tot_crashes_error'].abs()
matrix_test_results['crashes_per_100k_vmt_error'] = matrix_test_results['actual_crashes_per_100k_vmt'] - matrix_test_results['crashes_per_100k_vmt_input']
matrix_test_results['crashes_per_100k_vmt_abs_error'] = matrix_test_results['crashes_per_100k_vmt_error'].abs()


matrix_test_results.head()

# PLotting 

In [ ]:
error_plot(matrix_test_results, x_col='total_vmt', y_col='tot_crashes_error', split_methods=False, smooth_frac=0.3, lower_quant=0.10, upper_quant=0.90)
error_plot(matrix_test_results, x_col='expected_crashes', y_col='total_crashes', split_methods=False, smooth_frac=0.3, lower_quant=0.25, upper_quant=0.75)

# bunch of complicated stuff

In [ ]:
#Making the full test function that runs through a matrix of parameters and tests the cumulative crash function


def test_crash_matrix_cumulative(crash_matrix_dict):
    # Create a DataFrame of all parameter combinations
    def dict_combinations_to_df(param_dict):
        # Remove 'samples' from the dict and get its value (default to 1 if not present)
        samples = param_dict.pop('samples', 1)
        keys = list(param_dict.keys())
        combos = list(itertools.product(*param_dict.values()))
        # Repeat each combo 'samples' times
        combos = combos * samples
        crash_param_df = pd.DataFrame(combos, columns=keys)

        # Split 'method' into 'method_labol' '
        crash_param_df[['method', 'batch_seconds']] = pd.DataFrame(
            crash_param_df['method_spec'].tolist(), index=crash_param_df.index
        )
        crash_param_df = crash_param_df.drop(columns=['method_spec'])

        print(f'number of samples tested: {len(crash_param_df)}')
        print(f'unique samples : {len(crash_param_df.drop_duplicates())}')
        return crash_param_df

    # create the actual dataframe of combinations
    crash_param_df = dict_combinations_to_df(crash_matrix_dict)

    results = []
    METERS_PER_100K_MILES = 160_934_400.0
    M_PER_MILE = 1609.344

    for _, row in crash_param_df.iterrows():
        crash_counter = 0
        


        for _ in range(int(row['run_steps'])):
            crashes, state = should_crash_happen_cumulative(
                crashes_per_100k_vmt=float(row['crashes_per_100k_vmt']),
                num_cars=int(row['num_cars_per_step']),
                avg_speed_mps=float(row['avg_mps_of_step']),
               

                state=state
            )
            crash_counter += int(crashes)

        # Total VMT over the run (meters) = cars * m/s * seconds; seconds = run_steps
        total_vmt_m = float(row['num_cars_per_step']) * float(row['avg_mps_of_step']) * float(row['run_steps'])
        actual_crashes_per_100k_vmt = crash_counter / (total_vmt_m / METERS_PER_100K_MILES)
        expected_crashes = (float(row['crashes_per_100k_vmt']) / METERS_PER_100K_MILES) * total_vmt_m

        results.append({
            'method': method,
            'batch_seconds': (None if method!='batched_poisson' else (None if pd.isna(batch_seconds) else int(batch_seconds))),
            'actual_crashes_per_100k_vmt': actual_crashes_per_100k_vmt,
            'total_crashes': crash_counter,
            'total_vmt': total_vmt_m / M_PER_MILE,
            'expected_crashes': expected_crashes,
            # inputs
            'crashes_per_100k_vmt_input': float(row['crashes_per_100k_vmt']),
            'num_cars_input': int(row['num_cars_per_step']),
            'avg_mps_input': float(row['avg_mps_of_step']),
            'run_steps_input': int(row['run_steps']),
        })



    return pd.DataFrame(results)

In [ ]:
crash_matrix_dict = {
    "crashes_per_100k_vmt": [10, 20, 50],
    "num_cars_per_step":    [100, 300, 500],
    "avg_mps_of_step":      [10.0, 20.0, 30.0],
    "run_steps":            [3600, 7200, 14400],  # 1h, 2h, 4h at 1s steps
    "step_seconds":         [1.0],

    # Tie method + its parameter together
    "method": [
        ("poisson_clock", None),
        ("randomized_rounding", None),
        #("poisson", None),
        #("batched_poisson", 10)
    ],

    "samples": 3
}

# Run the crash matrix test function
matrix_test_results = test_crash_matrix_cumulative(crash_matrix_dict)

In [ ]:
def error_plot(df, x_col, y_col, lower_quant=0.25, upper_quant=0.75,
               split_methods=False, smooth_frac=0.2, iqr_color='blue'):
    df = df.copy()

    # Label that includes batch seconds if applicable
    if "batch_seconds" in df.columns:
        df["method_label"] = np.where(
            df["method"].eq("batched_poisson"),
            df["method"] + " (" + df["batch_seconds"].astype("Int64").astype(str) + "s)",
            df["method"]
        )
    else:
        df["method_label"] = df["method"]

    def _add_shading(ax, data):
        d = data.sort_values(x_col)
        # Guard: need at least a few points to compute quantiles & smooth
        if d.shape[0] < 5:
            return
        lower = d.groupby(x_col, as_index=False)[y_col].quantile(lower_quant)
        upper = d.groupby(x_col, as_index=False)[y_col].quantile(upper_quant)

        # Need matching x for fill_between; inner join on x
        q = lower.merge(upper, on=x_col, suffixes=('_lo', '_hi')).dropna()
        if q.shape[0] < 5:
            return

        sm_lo = lowess(q[f"{y_col}_lo"], q[x_col], frac=smooth_frac, return_sorted=False)
        sm_hi = lowess(q[f"{y_col}_hi"], q[x_col], frac=smooth_frac, return_sorted=False)
        ax.fill_between(q[x_col], sm_lo, sm_hi, color=iqr_color, alpha=0.2,
                        label=f'IQR ({int(100*lower_quant)}–{int(100*upper_quant)}%)')

    def _add_sse_box(ax, data):
        sse = float(np.square(data[y_col]).sum())
        txt = f"SSE = {sse:,.0f}"
        ax.text(0.02, 0.95, txt, transform=ax.transAxes, va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="gray"),
                fontsize=9)

    if not split_methods:
        plt.figure(figsize=(8, 4))
        ax = sns.scatterplot(
            data=df, x=x_col, y=y_col,
            hue="method_label", style="method_label",
            alpha=0.7, s=30, edgecolor=None
        )
        ax.axhline(0, color='black', linestyle='-', linewidth=1)
        _add_shading(ax, df)
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title('Absolute Error vs Total VMT by Method')
        ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
    else:
        g = sns.FacetGrid(df, col="method_label", sharey=True, sharex=False,
                          height=4, aspect=1.2, despine=True)

        def _facet(data, color, **kws):
            ax = plt.gca()
            sns.scatterplot(data=data, x=x_col, y=y_col, color=color,
                            alpha=0.7, s=30, edgecolor=None, ax=ax)
            ax.axhline(0, color='black', linestyle='-', linewidth=1)
            _add_shading(ax, data)
            _add_sse_box(ax, data)  # <- only called in split_methods=True path

        g.map_dataframe(_facet)
        g.set_axis_labels(x_col, y_col)
        g.set_titles(col_template="{col_name}")
        for ax in g.axes.flat:
            leg = ax.get_legend()
            if leg: leg.remove()
        plt.tight_layout()
        plt.show()






